# Topic 32 — Convolutional Neural Networks (CNNs)
### Theory → convolution by hand → sklearn digits with a real CNN → MNIST → image augmentation.

MLPs (Topic 30) flatten images into a long list of pixels, throwing away spatial structure
(neighboring pixels being related). **CNNs** instead slide small learnable filters over the image,
preserving spatial relationships and dramatically reducing the number of parameters needed.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("device:", device)

## 1. Convolution, kernel/filter, feature map — from scratch on a tiny example

A **kernel** (filter) is a small grid of learnable numbers (e.g. 3x3). Sliding it across an image
and computing a weighted sum at each position produces a **feature map** — each kernel learns to
detect one specific pattern (an edge, a corner, a texture).

In [ ]:
def convolve2d(image, kernel):
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh, ow = ih - kh + 1, iw - kw + 1   # output shrinks by kernel_size - 1 (no padding, stride 1)
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(region * kernel)
    return output

# A simple toy image: a vertical edge (left half dark, right half bright)
image = np.array([
    [0, 0, 1, 1],
    [0, 0, 1, 1],
    [0, 0, 1, 1],
    [0, 0, 1, 1],
], dtype=float)

# A vertical-edge-detecting kernel
vertical_edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
])

feature_map = convolve2d(image, vertical_edge_kernel)
print("input image:\n", image)
print("\nfeature map (high value = edge detected):\n", feature_map)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(image, cmap="gray"); axes[0].set_title("input image")
axes[1].imshow(feature_map, cmap="gray"); axes[1].set_title("feature map (edges highlighted)")
plt.show()

## 2. Stride and padding

- **Stride**: how many pixels the kernel moves each step. stride=1 = move one pixel at a time
  (dense, larger output). stride=2 = skip every other position (smaller, faster output).
- **Padding**: adding a border of zeros around the image before convolving, so the output stays the
  same size as the input instead of shrinking (`padding="same"` in PyTorch, or a specific pixel count).

In [ ]:
def convolve2d_strided(image, kernel, stride=1, padding=0):
    if padding > 0:
        image = np.pad(image, padding, mode="constant")
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = (ih - kh) // stride + 1
    ow = (iw - kw) // stride + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            output[i, j] = np.sum(region * kernel)
    return output

out_stride1 = convolve2d_strided(image, vertical_edge_kernel, stride=1, padding=0)
out_stride2 = convolve2d_strided(image, vertical_edge_kernel, stride=2, padding=0)
out_padded = convolve2d_strided(image, vertical_edge_kernel, stride=1, padding=1)

print("stride=1, no padding -> output shape:", out_stride1.shape)
print("stride=2, no padding -> output shape:", out_stride2.shape)
print("stride=1, padding=1  -> output shape:", out_padded.shape, "(matches input size!)")

## 3. Pooling — downsampling feature maps

**Max pooling** takes the max value in each small window, shrinking the feature map while keeping
the strongest activations. Makes the network more robust to small shifts/distortions in the image
and reduces computation for later layers.

In [ ]:
def max_pool2d(feature_map, pool_size=2, stride=2):
    ih, iw = feature_map.shape
    oh = (ih - pool_size) // stride + 1
    ow = (iw - pool_size) // stride + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = feature_map[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(region)
    return output

pooled = max_pool2d(feature_map, pool_size=2, stride=2)
print("before pooling:\n", feature_map)
print("\nafter 2x2 max pooling:\n", pooled)

## 4. Receptive field & flattening

- **Receptive field**: how much of the ORIGINAL image a single output unit "sees", after passing
  through several conv/pool layers. It grows with each layer — deeper layers see progressively
  larger, more abstract regions of the input.
- **Flattening**: after several conv+pool layers, the resulting 3D feature maps (height x width x
  channels) get reshaped into one long vector before being fed into final `Linear` layers for
  classification — exactly like an MLP's input.

## 5. Building a real CNN in PyTorch on sklearn digits

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.images.astype(np.float32) / 16.0   # normalize pixel values to [0,1]
y = digits.target

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# CNNs expect shape (batch, channels, height, width) -- add a channel dimension (1 = grayscale)
X_train_t = torch.tensor(X_train).unsqueeze(1)
X_val_t = torch.tensor(X_val).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
print("input shape:", X_train_t.shape, "-> (batch, channels, height, width)")

train_loader = DataLoader(torch.utils.data.TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
val_loader = DataLoader(torch.utils.data.TensorDataset(X_val_t, y_val_t), batch_size=32, shuffle=False)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        # After two 2x2 poolings, an 8x8 image becomes 2x2, with 32 channels -> 32*2*2 = 128
        self.fc1 = nn.Linear(32 * 2 * 2, 64)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 8x8 -> 4x4
        x = self.pool(self.relu(self.conv2(x)))   # 4x4 -> 2x2
        x = x.flatten(start_dim=1)                # flatten -- same idea as Part 4 above
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

cnn_model = SimpleCNN().to(device)
print(cnn_model)
print("\ntotal parameters:", sum(p.numel() for p in cnn_model.parameters()))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

def accuracy(loader, model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            correct += (model(Xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    return correct / total

history = {"train_loss": [], "val_acc": []}
for epoch in range(15):
    cnn_model.train()
    losses = []
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(cnn_model(Xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    history["train_loss"].append(np.mean(losses))
    history["val_acc"].append(accuracy(val_loader, cnn_model))
    print(f"epoch {epoch+1}: loss={history['train_loss'][-1]:.3f}  val_acc={history['val_acc'][-1]:.3f}")

plt.figure(figsize=(5, 4))
plt.plot(history["val_acc"])
plt.xlabel("epoch"); plt.ylabel("val accuracy")
plt.title("CNN validation accuracy over training")
plt.show()

## 6. Image data augmentation with `torchvision.transforms`

Randomly rotating/shifting/flipping training images each epoch effectively multiplies your dataset
size and reduces overfitting (Topic 31) — the model can't just memorize exact pixel arrangements.

In [ ]:
augmentation = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
])

# Demonstrate on one digit image
sample_img = torch.tensor(digits.images[0] / 16.0, dtype=torch.float32).unsqueeze(0)   # (1, 8, 8)

fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
axes[0].imshow(sample_img.squeeze(), cmap="gray"); axes[0].set_title("original")
for i in range(1, 5):
    augmented = augmentation(sample_img)
    axes[i].imshow(augmented.squeeze(), cmap="gray")
    axes[i].set_title(f"augmented {i}")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()
# Each "augmented" version is a slightly rotated/shifted digit -- still recognizably the same digit,
# giving the model more visual variety to learn from without collecting new data.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Change conv1's out_channels from 16 to 32 and conv2's from 32 to 64 -- how do parameter
#    count and val accuracy change?
# 2. Add a 3rd conv+pool block -- note you'll need to adjust the flatten size math accordingly.
# 3. Wrap the augmentation transform into the training loop (apply it to each batch before
#    forward()) and compare val accuracy after 15 epochs to the non-augmented run above.
# 4. In one sentence: why does a CNN typically need far FEWER parameters than an MLP to achieve
#    similar accuracy on image data (hint: look at the conv1/conv2 parameter counts vs an
#    equivalent-sized nn.Linear layer)?

---
### Next up: **Topic 33 — RNNs** (back to sequential/text data).

Say "next" when you're ready.